# Tune the planner, step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create a demo table with correlated columns, measure the planner's bad row estimate, create extended statistics, and prove the fix.

Each step shows the plain SQL too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the bank is loaded on your branch.

## Setup — connect

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env

DEMO_TABLE = 'account_perf_demo'
DEMO_STAT = 'account_perf_stat'
SCHEMA = _env('LB_SCHEMA', 'bank')

def _explain(cur, query: str) -> tuple[int, int]:
    cur.execute(f"EXPLAIN (ANALYZE, FORMAT JSON) {query}")
    raw = cur.fetchone()[0][0]
    p = raw['Plan']
    while p.get('Plans'):
        p = p['Plans'][0]
    est = p.get('Plan Rows', 0)
    act = p.get('Actual Rows', 0)
    return est, act

print('Ready.')

## Step 1 — Create demo table

Create a table where region is 100 percent correlated with branch_id. Branch BR-CDMX is MX-Centro, BR-MTY is MX-Noreste.

```sql
CREATE TABLE account_perf_demo (
    id BIGSERIAL PRIMARY KEY,
    account_id TEXT,
    branch_id TEXT,
    region TEXT,
    amount NUMERIC,
    health_pct NUMERIC
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{DEMO_TABLE} CASCADE")
        cur.execute(f"DROP STATISTICS IF EXISTS {SCHEMA}.{DEMO_STAT}")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{DEMO_TABLE} (
                id BIGSERIAL PRIMARY KEY,
                account_id TEXT,
                branch_id TEXT,
                region TEXT,
                amount NUMERIC,
                health_pct NUMERIC
            )
        """)
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{DEMO_TABLE}(account_id, branch_id, region, amount, health_pct)
            SELECT
                'CTA-' || g,
                (ARRAY['BR-CDMX', 'BR-CDMX', 'BR-MTY', 'BR-MTY'])[1 + (g % 4)],
                CASE (g % 4)
                    WHEN 0 THEN 'MX-Centro'
                    WHEN 1 THEN 'MX-Centro'
                    WHEN 2 THEN 'MX-Noreste'
                    ELSE 'MX-Noreste'
                END,
                50.0 + (g % 30),
                80.0 + (g % 20)
            FROM generate_series(1, 100000) g
        """)
        cur.execute(f"ANALYZE {SCHEMA}.{DEMO_TABLE}")
        print(f'✓ Created {SCHEMA}.{DEMO_TABLE} (100k rows, correlated columns)')

## Step 2 — Bad estimate (no statistics)

Query with a correlated filter. Planner assumes independence and underestimates rows.

```sql
EXPLAIN ANALYZE
SELECT account_id FROM account_perf_demo
WHERE branch_id = 'BR-CDMX' AND region = 'MX-Centro'
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        query = f"SELECT account_id FROM {SCHEMA}.{DEMO_TABLE} WHERE branch_id = 'BR-CDMX' AND region = 'MX-Centro'"
        est, act = _explain(cur, query)
        if est > 0:
            ratio = act / est
            print(f'Before CREATE STATISTICS:')
            print(f'  Estimated: {est:,}')
            print(f'  Actual:    {act:,}')
            print(f'  Error:     {ratio:.1f}x off')

## Step 3 — Create statistics

Teach the planner about the correlation.

```sql
CREATE STATISTICS account_perf_stat
ON branch_id, region
FROM account_perf_demo;

ANALYZE account_perf_demo;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            CREATE STATISTICS {SCHEMA}.{DEMO_STAT}
            ON branch_id, region
            FROM {SCHEMA}.{DEMO_TABLE}
        """)
        cur.execute(f"ANALYZE {SCHEMA}.{DEMO_TABLE}")
        print(f'✓ Created statistics; planner now knows the correlation')

## Step 4 — Fixed estimate (with statistics)

Same query. Now the planner estimates accurately.

```sql
EXPLAIN ANALYZE
SELECT account_id FROM account_perf_demo
WHERE branch_id = 'BR-CDMX' AND region = 'MX-Centro'
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        query = f"SELECT account_id FROM {SCHEMA}.{DEMO_TABLE} WHERE branch_id = 'BR-CDMX' AND region = 'MX-Centro'"
        est, act = _explain(cur, query)
        print(f'After CREATE STATISTICS:')
        print(f'  Estimated: {est:,}')
        print(f'  Actual:    {act:,}')
        print(f'  Accurate: {est == act}')

## What you learned

Extended statistics teach the planner about column correlations. Without them, the planner assumes independence and badly underestimates rows, which cascades into bad execution plans. Same table, same data, same query — the difference is the statistics.

Next: **One engine, every index** measures how different index types are selected for different queries.